<img src="logo.png" alt="Vegeta" width="240">

# NISUS — the observation drone and its computer-vision sibling (31)

**Nisus-OBS** is a remotely piloted observation aircraft: a camera and a live 5.8 GHz video link, a flight controller
with stabilisation and return-to-home, no onboard computer. **Nisus-Zero** is the same aircraft with an NVIDIA Jetson
for onboard computer vision, its own camera, power branch and data link. Both are the same airframe: a continuous
1.4 m straight-tapered wing on a rounded central pod, one electric pusher propeller between two carbon tail booms, an
H-tail (a stabiliser joining the booms, a fin with rudder on each). No folding wings.

This notebook is the **first engineering approximation**: every number is calculated, sourced or labelled as an
assumption, and later iterations refine it with calculations, simulations and measurements. Nothing runs by itself:
the engineer runs each cell. Missing analyses say **NOT RUN**.

| part | what | where the code is |
|---|---|---|
| 1 | requirements, starting dimensions and what changed | `designs/nisus.py` |
| 2 | CAD (Dedalus): parts, exports, three-view, exploded view, propeller disk, packaging | `nisus.py`, `nisus_drawings.py` |
| 3 | materials, construction, joints and load paths, transport | `nisus_systems.py`, `nisus_structure.py` |
| 4 | mass budgets of both variants, CG, inertia, wing loading | `nisus_systems.py` |
| 5 | electronics, power branches and their energy | `nisus_systems.py` |
| 6 | propulsion: requirement, shortlist, selection (Boreas), ESC, battery | `nisus_systems.py` |
| 7 | aerodynamics: vortex lattice, derivatives, whole-aircraft CFD (Aeromant / OpenFOAM) | `nisus_flight.py`, `nisus_cfd.py` |
| 8 | structure: load cases, hand checks, FEA (Talos / CalculiX), mesh convergence | `nisus_structure.py` |
| 9 | flight: trim, CG range, control, stall, turns, gusts, pull-up, approach, limits | `nisus_flight.py` |
| 10 | mission energy: phases, return reserve, battery iteration, survey time and radius | `nisus_systems.py` |
| 11 | MuJoCo (Chiron) six-degree-of-freedom missions with videos | `nisus_robot.py`, `nisus_controller.py`, `nisus_scenario.py` |
| 12 | the first recommended build of OBS and Zero; what to refine next | — |

Switches (environment variables): `VEGETA_DEBUG=1` mocks FEA and CFD (a dry run of every cell); `NISUS_QUICK=1` flies
only the two calm missions and renders small movies; `VEGETA_SKIP_OPENFOAM=1` skips the CFD (NOT RUN);
`NISUS_FEA=0` skips the FEA (NOT RUN); `NISUS_CFD=screening` runs the CFD without the fine mesh.

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display
from vegeta import dedalus, talos, aeromant, boreas, cache
from vegeta.talos import viz as tviz
sys.path.insert(0, "designs")
import nisus, nisus_systems as ns, nisus_flight as nf, nisus_structure as nst, nisus_cfd as ncfd, nisus_drawings as nd
import nisus_robot as nr, nisus_controller as nc, nisus_scenario as nsc

pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
ROOT = Path("_runs/nisus"); ROOT.mkdir(parents=True, exist_ok=True)
OUT = Path("output") / "31_nisus"; OUT.mkdir(parents=True, exist_ok=True)       # drawings, CAD, tables, telemetry, movies
for sub in ("cad", "drawings", "tables", "telemetry", "plots"):
    (OUT / sub).mkdir(exist_ok=True)
cache.notebook("31_nisus")
QUICK = os.environ.get("NISUS_QUICK") == "1"
RUN_FEA = os.environ.get("NISUS_FEA", "1") != "0"
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
CFD_FINE = os.environ.get("NISUS_CFD", "full") != "screening"
G, RHO = ns.G, ns.RHO
VARIANTS = ("OBS", "Zero")
COLOR = {"OBS": "#1f77b4", "Zero": "#d62728"}
STATUS = {}                                 # what ran, what is NOT RUN — collected for the summary at the end
print(f"DEBUG {DEBUG} | QUICK {QUICK} | FEA {'on' if RUN_FEA else 'NOT RUN'} | CFD {'on' if RUN_CFD else 'NOT RUN'}{' (fine mesh too)' if CFD_FINE else ''}")
def save_table(df, name):
    df.to_csv(OUT / "tables" / f"{name}.csv")
    return df

# Part 1 — requirements and the starting shape

The brief fixes the shape and the main dimensions: a continuous straight-tapered wing of 1400 mm span with 230/170 mm
chords (0.28 m², aspect ratio 7, the pod-covered part included), a rounded central pod of about 450 × 100 × 110 mm,
one 9-inch pusher propeller centred between two ~580 mm tail booms 320 mm apart, a 400 × 140 mm horizontal tail
joining them and two fins with rudders; about 900 mm nose to tail. Cruise 14–18 m/s; a ten-minute airborne survey
including the return as the first sizing target; 20 % of the available energy left after landing.

**First choices** (each one a starting point for the next iteration):

| item | choice | why |
|---|---|---|
| wing section | NACA 3412 | 12 % gives depth for a 10 mm spar and a stiff foam core; 3 % camber puts the best L/D near the survey CL (0.3–0.6) at Re ≈ 2·10⁵ |
| wing incidence | 1.5° to the pod axis | the pod flies level (least drag) near 15 m/s |
| dihedral | 3° | with the high wing on the pod, enough spiral stability for a hand-flown OBS; little loss in roll rate |
| ailerons | outer 45 % of each half span, 25 % chord | roll authority (p b/2V ≈ 0.07–0.09) with a 9 g-class servo in each wing |
| tail | 400 × 140 mm H-tail at the booms' height, NACA 0005, incidence 1.5°, 35 % elevator | tail volume V_h ≈ 0.52 (0.4–0.6 is usual for a stable trainer-class aircraft) |
| fins | two 140 × 140 mm fins above the booms + 60 mm ventral part each, 35 % rudders, one servo for both | V_v ≈ 0.067; the ventral parts are the tail bumpers the aircraft rests on |
| tail moment arm | 0.52 m (wing ac to tail ac) | from the boom length and the reconciliation below |
| spar | 10/8 mm pultruded carbon tube at 30 % chord, 1.0 m, 6/4 tip rods | the bending stiffness and strength of the wing in one purchasable part |
| boom | 10/8 mm carbon tube | the 10 mm starting reference; checked below (stress, deflection, frequency, propeller clearance) |
| motor | SunnySky X2216 V3 KV1250 + APC 9×6E on 3S, 6° down-thrust | calculated and selected in Part 6 |

In [ ]:
D = nisus.Nisus()
P = nisus.resolve()
L = nisus.Nisus.layout(P)
start = {"span [mm]": (1400, P["span"]), "root / tip chord [mm]": ("230 / 170", f"{P['root_chord']:.0f} / {P['tip_chord']:.0f}"),
         "wing area [m²]": (0.28, round(L["S_ref"], 4)), "aspect ratio": (7, round(L["AR"], 2)),
         "pod length × width × height [mm]": ("450 × 100 × 110", f"{P['pod_length']:.0f} × {P['pod_width']:.0f} × {P['pod_height']:.0f}"),
         "nose ahead of the wing [mm]": ("—", P["nose_length"]),
         "boom length incl. socket [mm]": (580, P["boom_length"]), "boom spacing [mm]": (320, 2 * P["boom_y"]), "boom tube [mm]": ("Ø10", f"{P['boom_od']:.0f}/{P['boom_id']:.0f}"),
         "horizontal tail [mm]": ("400 × 140", f"{P['tail_span']:.0f} × {P['tail_chord']:.0f}"), "tail area [m²]": (0.056, round(L["S_h"], 4)),
         "propeller [mm]": (228.6, P["prop_diameter"]), "overall length [mm]": ("~900", round(L["overall_length"])),
         "lateral propeller clearance [mm]": (40.7, round(L["prop_clearance_boom"], 1))}
display(pd.DataFrame(start, index=["brief", "first approximation"]).T)
print("Reconciliation: with a 450 mm pod and its nose 180 mm ahead of the wing, the battery could not move far enough forward —")
print("the centre of gravity of the loaded aircraft sat near 60 % MAC with the pack against the bay's front (behind the neutral point:")
print("unstable). The pod grew to 550 mm with the nose 300 mm ahead of the wing (the pack and, on Zero, the Jetson ahead of the spar);")
print(f"the booms shrank 30 mm to keep the tail where it was. Overall length {L['overall_length']:.0f} mm instead of ~900: the price of a balanced aircraft.")
print(f"The thrust line rose to the pod's top line (z = {P['motor_z']:.0f} mm) so the propeller clears the ground at rest on the keel and the tail bumpers;")
print(f"the boom-to-disc clearance is {L['prop_clearance_boom']:.1f} mm nominal (the brief's 40.7 mm with the thrust line at the booms' height).")

# Part 2 — CAD (Dedalus)

One parametric design (`designs/nisus.py:Nisus`, a subclass of notebook 09's `FixedWing`) builds the whole aircraft and
every part from the same numbers the analyses use (`Nisus.layout`). The parts: wing, main spar, rear carry-through
tube, pod (printed shell), nose cone (replaceable), booms, boom root fittings (printed, clamped on both tubes), tail
(stabiliser and fins), tail fittings, motor mount, electronics tray, battery tray, belly keel skid.

In [ ]:
display(pd.DataFrame(D.params.table()).set_index("name").head(70))
GEOM = {}
for part in nisus.PARTS:
    g = D.generate(part=part)
    GEOM[part] = g
m = {k: g.measure() for k, g in GEOM.items()}
parts = pd.DataFrame({k: {"valid": v["valid"] if "valid" in v else GEOM[k].shape.isValid(), "solids": len(GEOM[k].shape.Solids()),
                          "volume [cm³]": (v["volume"] or 0) / 1000, "surface [cm²]": v["surface_area"] / 100,
                          "bounding box [mm]": " × ".join(f"{d:.0f}" for d in GEOM[k].dimensions)} for k, v in m.items()}).T
save_table(parts, "cad_parts")

In [ ]:
# STEP and STL of the aircraft and of every part (the same files the FEA and CFD read)
for part, g in GEOM.items():
    g.export_step(str(OUT / "cad" / f"nisus_{part}.step"))
    g.export_stl(str(OUT / "cad" / f"nisus_{part}.stl"), tolerance=0.2)
D.generate(part="aircraft", show_prop=True).export_stl(str(OUT / "cad" / "nisus_aircraft_with_prop_disc.stl"), tolerance=0.3)
print(len(list((OUT / "cad").glob("*.step"))), "STEP and", len(list((OUT / "cad").glob("*.stl"))), "STL files in", OUT / "cad")
try:
    from vegeta.dedalus import viz as dviz
    dviz.show(dviz.plot3d(D.generate(part="aircraft", show_prop=True)))
except Exception as e:
    print("3D view needs pyvista and a display:", type(e).__name__, e)

In [ ]:
fig = nd.three_view(P); fig.savefig(OUT / "drawings" / "three_view.png", dpi=150); plt.show()

In [ ]:
try:
    nd.exploded_png(OUT / "drawings" / "exploded.png")
    display(Image(str(OUT / "drawings" / "exploded.png"), width=1000))
except Exception as e:
    print("exploded view needs pyvista with an off-screen renderer (xvfb-run on a headless machine):", type(e).__name__, e)

## The propeller's swept disk

The disc turns between the booms, behind the wing and the pod, ahead of the stabiliser. The clearances below are for
the **whole swept disk**, not a blade position: to the booms (nominal, then with the boom's lateral deflection under
the fin load and ±1.5 mm build tolerance), to the wing's trailing edge and the tail (axial), to the ground at rest and
at a level belly touchdown.

In [ ]:
CLEAR, fig = nd.prop_clearance(P)
fig.savefig(OUT / "drawings" / "propeller_disk.png", dpi=130); plt.show()
display(save_table(CLEAR, "propeller_clearance"))
BOOM = nisus.boom_check(P, tail_load_N=nst.ULTIMATE * 0.5 * 21.5, fin_side_load_N=nst.ULTIMATE * 9.7)
display(pd.Series({k: v for k, v in BOOM.items() if not isinstance(v, dict)}, name="boom tube by hand (ultimate tail and fin loads)").to_frame())

## Packaging: what goes where inside the pod

The pod's inner section is 97.6 × 107.6 mm (1.2 mm LW-PLA walls). Five bays from the nose: the camera(s) in the
replaceable nose cone; the battery bay (the pack slides along it on its tray to set the CG); the computer bay (Zero's
Jetson stands on its side beside the pack; OBS puts its flight-controller tray there); the flight-controller tray
under the wing's front (Zero); the video transmitter, ESC, wiring and the cooling vents in the tail cone. The hatch is
the top of the pod ahead of the wing, held by two magnets and a latch; the electronics trays slide out on rails; the
nose cone is a bayonet with the cameras on it (a broken nose is replaced, not repaired).

In [ ]:
for v in VARIANTS:
    fig = nd.internal_layout(v); fig.savefig(OUT / "drawings" / f"internal_{v}.png", dpi=130); plt.show()
bays = nisus.Nisus.bays(P)
fit_rows = {}
for b in ns.BATTERIES:
    f = ns.battery_fits(b)
    fit_rows[b.name] = {"pack [mm]": " × ".join(f"{d:g}" for d in f["pack [mm]"]), "bay [mm]": " × ".join(f"{d:.0f}" for d in f["bay [mm]"]), "fits": f["fits"]}
display(pd.DataFrame(fit_rows).T)
jet = ns.JETSON_OPTIONS["nano_b01"]["size_mm"]
x0, x1, w, h = bays["computer bay (Zero: the Jetson on its side beside the battery)"]
print(f"Jetson Nano B01 ({jet[0]}×{jet[1]}×{jet[2]} mm) on its side in the computer bay ({x1 - x0:.0f} long, {w:.0f} wide, {h:.0f} high): "
      f"needs {jet[0]} long, {jet[2]} + 34 (the pack beside it) = {jet[2] + 34} wide, {jet[1]} high -> "
      f"{'fits' if jet[0] <= x1 - x0 and jet[2] + 34 <= w - 6 and jet[1] <= h - 10 else 'DOES NOT FIT'}; "
      f"its fan draws through the nose vents and exhausts into the tail cone's vents (the ESC's airflow)")

# Part 3 — materials, construction and load paths

**Materials by function.** Foam for aerodynamic volume (XPS 30 kg/m³ hot-wire-cut wing and tail cores, lightened aft
of the spar), balsa for light stiffness (a 1.5 mm D-box from the leading edge to the spar over the inner 60 % of the
span: it carries most of the wing's torsion), carbon tubes where bending is concentrated (main spar, rear
carry-through tube, booms, stabiliser spar), printed plastic for fittings, mounts and complex shapes (PETG for the
load-carrying fittings, LW-PLA for the pod shell, TPU for the keel skid), laminating film over the foam.

**Load paths.** Lift: foam core and D-box → main spar → the pod's wing saddle (a printed PETG keel frame with two M5
nylon bolts). Tail loads: stabiliser and fins → tail sleeves → booms → the boom root fittings, which **clamp both the
main spar and the rear carry-through tube**: the boom's moment reaches the wing as a couple between the two tubes
(not as peel through the foam — the first draft bonded the fitting to the wing's lower skin and its hand check failed
at −62 %), and the tubes take it to the saddle. Thrust: propeller → motor → printed mount (four M3 screws in heat-set
inserts) → the pod's tail ring → keel frame. Battery: tray + strap + velcro → keel frame.

**Transport.** No folding: the wing, booms and tail stay together as one rigid unit that lifts off the pod (two M5 nylon
bolts, one multi-pin servo connector); the package is 1400 × ~720 × 280 mm (wing unit) plus the 550 mm pod — a car's
back seat. The bolts are the designed weak link in a cartwheel.

In [ ]:
mat = pd.DataFrame({k: {kk: vv for kk, vv in v.items()} for k, v in ns.MATERIALS.items()}).T
display(save_table(mat, "materials"))
plist = []
for item, grp, g, x, y, z, basis in ns.structure_items(P):
    plist.append({"part": item, "group": grp, "mass [g]": round(g, 1), "basis": basis})
plist = pd.DataFrame(plist).set_index("part")
display(save_table(plist, "structure_part_list"))
display(pd.Series(nst.PRINT_SETTINGS, name="print orientation, walls, infill").to_frame())

# Part 4 — mass budgets, centre of gravity, inertia

Each variant's mass table is the structure from the CAD (volumes × densities, print fill, covering) plus its bought
components (`ns.COMPONENTS`, each with mass, place, power, price, source and URL) plus the pack. The battery is placed
along its bay so the CG sits at 30 % MAC (as a builder trims a model); the table says when the bay's ends prevent it.
The earlier camera-free budget (1100 g) is the reference column; the brief's first Zero estimate was ~1330 g.

In [ ]:
TABLES, CI = {}, {}
for v in VARIANTS:
    t = ns.mass_table(v); TABLES[v] = t; CI[v] = ns.cg_inertia(t)
    save_table(t, f"mass_table_{v}")
cmp_ = pd.concat([ns.budget_comparison(v)[[f"{v} [g]"]] for v in VARIANTS], axis=1)
cmp_.insert(0, "reference (camera-free) [g]", ns.budget_comparison("OBS")["reference [g]"])
display(save_table(cmp_.round(1), "mass_budget_comparison"))
ci = pd.DataFrame({v: {k: x for k, x in CI[v].items() if k != "inertia"} for v in VARIANTS})
display(save_table(ci, "cg_inertia"))
for v in VARIANTS:
    print(f"Nisus-{v}: {CI[v]['mass_kg'] * 1000:.0f} g, CG at {100 * CI[v]['x_cg_frac_mac']:.1f} % MAC, wing loading {CI[v]['wing_loading_g_dm2']:.1f} g/dm², "
          f"battery at x = {TABLES[v].loc[TABLES[v].attrs['battery_item'], 'x [mm]']:.0f} mm  {TABLES[v].attrs['ballast_note']}")
print(f"Zero against the brief's first estimate (1330 g): {CI['Zero']['mass_kg'] * 1000 - 1330:+.0f} g — the data link (25 g), the heavier structure "
      f"(the rear tube, the D-box, the clamp fittings) and the longer pod, less the lighter servos/FC/allowance")
display(TABLES["Zero"])

In [ ]:
comp = pd.concat({v: ns.component_table(v) for v in VARIANTS}, names=["variant"])
display(save_table(comp, "components_with_sources"))
print("Component sources: designs/data/nisus_component_sources.md (market research of 2026-10-09: search-index snippets of the cited pages;"
      " no page was opened and NO availability is confirmed — re-verify each line before ordering)")

# Part 5 — electronics and their energy

**Power branches** (all downstream of the flight controller's 110 A current sensor on the main battery lead, so the
battery current it logs includes every branch): (1) the ESC and motor; (2) the Kakute's own 5.3 V BEC (3 A) for the
GNSS, receiver and telemetry radio; (3) its servo rail (5.3/7.2 V, 3 A) for the four servos; (4) the video transmitter
directly on the battery (7–36 V), which powers the pilot's camera from its 5 V output; (5) on Zero, a separate
5 V / 5 A buck with its own fuse for the Jetson, its camera, fan and data link. The flight controller never depends on
branch 5: a Jetson crash, reboot or brown-out leaves the aircraft flying (Part 11 simulates it). Servo peaks (stall) are
listed apart from their averages; the prelaunch ground operation (cameras and computer on, GNSS fix) is a phase of
its own.

In [ ]:
for v in VARIANTS:
    e = ns.electrical_loads(v)
    print(f"Nisus-{v}"); display(save_table(e, f"electrical_loads_{v}"))
pp = pd.concat({v: ns.phase_power(v)["electronics battery-side [W]"] for v in VARIANTS}, axis=1)
display(save_table(pp.round(2), "electronics_phase_power"))
for v in VARIANTS:
    print(f"Nisus-{v} regulated branches:"); display(ns.regulator_check(v).round(2))
display(save_table(ns.servo_check(P).round(2), "servo_check"))
print(f"Zero's vision installation: {ns.JETSON_INSTALLATION_W:g} W battery side (the brief's estimate for the original Nano 4 GB installation,"
      f" camera, cooling and conversion included; sensitivity {ns.JETSON_INSTALLATION_RANGE_W}); the data link is added separately (2 W at 5 V).")
display(pd.DataFrame(ns.JETSON_OPTIONS).T)

# Part 6 — propulsion: calculate first, then select

The requirement comes from the aircraft (the heavier Zero), the build-up polar and the targets: cruise 16 m/s (level:
thrust = drag), climb 3 m/s at 13 m/s, a 25 m/s dash, a climbing turn at n = 1.4 with a 30 % gust margin, and a hand
launch (static thrust ≥ 0.45 of the weight). Static thrust, in-flight thrust, shaft power (÷ η_prop) and electrical
input (÷ η_motor η_ESC = 0.80 × 0.96) are separate columns.

In [ ]:
BU = nisus.drag_buildup(P, 16.0)
A0 = nf.aero(P)
OSW = 0.9 * A0["e"]
print(f"build-up Cd0 = {BU['cd0']:.4f} (Re on the MAC {BU['re_wing']:.2e}); lattice e = {A0['e']:.3f} → Oswald {OSW:.3f}")
display(pd.Series({k: v * 1e4 for k, v in BU["parts_m2"].items()}, name="Cd0·S [cm²]").to_frame())
REQ = ns.requirements(CI["Zero"]["mass_kg"], BU["cd0"], L["AR"], OSW, L["S_ref"])
display(save_table(REQ.round(2), "propulsion_requirement"))
display(save_table(ns.motor_shortlist_table(REQ.attrs["T_max_required_N"]), "motor_shortlist"))

**Selection.** The SunnySky X2216 V3 KV1250 with an APC 9×6E on 3S is the only shortlist entry with a published 3S
point on this propeller (retailer copies of the maker's table: 20.8 A, 231 W, ~10.2 N static at 11.1 V) — about
0.7 of the weight in static thrust against the 0.45 asked, at 69 g. The T-Motor AT2312 (60 g) has no 3S data; the
AT2814 is 38 g heavier for power it does not need. The motor model below is Boreas' first-order motor with the KV
published, the winding resistance fitted to that static point and the no-load current assumed; the propeller is
Boreas BEMT on a generic 9×6 planform, scaled to the published static thrust (`thrust_correction`).
**ESC**: a 40 A AM32 2–4S with a 5 V/2 A BEC (14 g; the BEC is not used) — 40 A against a 22 A static peak; the
Hobbywing Skywalker 40A V2 (36 g) is the proven fallback.

In [ ]:
MOTOR, FIT = ns.x2216_model()
display(pd.Series(FIT, name="X2216 fitted to the published static point").to_frame())
PM = ns.propulsion_map(rebuild=True)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for V in (0.0, 10.0, 16.0, 22.0):
    tb = PM.table(V)
    ax[0].plot(tb.index, tb["thrust [N]"], label=f"{V:g} m/s"); ax[1].plot(tb.index, tb["electrical [W]"]); ax[2].plot(tb.index, tb["current [A]"])
for a_, lab in zip(ax, ("thrust [N]", "electrical power [W]", "motor current [A]")):
    a_.set_xlabel("rpm"); a_.set_ylabel(lab); a_.grid(alpha=0.3)
ax[0].legend(); ax[2].axhline(40, color="r", ls="--", lw=0.8); fig.tight_layout(); fig.savefig(OUT / "plots" / "propulsion_map.png", dpi=120); plt.show()
PTS = ns.propulsion_points(PM, CI["Zero"]["mass_kg"], BU["cd0"], L["AR"], OSW, L["S_ref"])
display(save_table(PTS.T, "propulsion_points"))

In [ ]:
display(save_table(ns.battery_table(), "batteries"))

# Part 7 — aerodynamics

The **vortex lattice** (PEREGRINE's, `peregrine_flight.vlm`) on the wing and the stabiliser gives the longitudinal
derivatives; the lateral-directional ones are analytic (tail-volume and strip-theory estimates) with the pod's terms
assumed. Every derivative carries its tag: **calculated (lattice)**, **calculated (analytic)**, **assumed**. The
lattice is attached flow only: the stall is a section value (CL_max = 0.9 × 1.30, assumed).

In [ ]:
POL = nf.polar(P, np.arange(-6, 15, 2))
display(save_table(POL.round(4), "polar_lattice"))
DER = {v: nf.derivatives(P, x_cg_m=CI[v]["x_cg_m"], z_cg_m=CI[v]["z_cg_m"]) for v in VARIANTS}
display(save_table(DER["Zero"], "derivatives_Zero"))
save_table(DER["OBS"], "derivatives_OBS")
print(f"neutral point {100 * (A0['x_np'] * 1000 - L['x_mac_le']) / L['mac']:.1f} % MAC; static margin OBS {100 * DER['OBS'].loc['static_margin', 'value']:.1f} %, "
      f"Zero {100 * DER['Zero'].loc['static_margin', 'value']:.1f} % MAC; downwash gradient {A0['deps_dalpha']:.2f}")

## Whole-aircraft CFD (Aeromant / OpenFOAM)

Steady RANS k-ω SST with wall functions, ISA sea level (ρ = 1.225 kg/m³, ν = 1.46·10⁻⁵ m²/s), fully turbulent (no
transition model: at Re ≈ 1.6–2.7·10⁵ on the MAC the real boundary layers are partly laminar, so friction is
over-predicted and laminar separation is not captured), velocity inlet, pressure outlet, slip sides, no-slip body, the
domain 5 L_ref upstream and to the sides, 12 behind (L_ref = span/4). **Screening** (2 cells/L_ref, levels 4/3/2, 300
iterations) gives the α and speed sweeps, the wing alone (interference) and the propeller running (rotor disk); the
**fine** mesh at the cruise point (3 cells/L_ref, levels 5/4/2, 600 iterations) says how far the screening is from
converged. Attached-flow RANS at a few angles does not predict the stall or the recovery.

In [ ]:
CRUISE_RPM = float(PTS.loc["cruise (T = D)", "rpm"])
CFD_CASES = ncfd.plan(ROOT / "cfd", x_cg_m=CI["Zero"]["x_cg_m"], z_cg_m=CI["Zero"]["z_cg_m"], cruise_rpm=CRUISE_RPM, fine=CFD_FINE)
for name, c in CFD_CASES.items():
    print(f"{name:38s} {c.template_name if hasattr(c, 'template_name') else ''} {c.workdir}")
if RUN_CFD:
    names = list(CFD_CASES)
    res = aeromant.run_cases([CFD_CASES[n] for n in names], jobs=1, processors=4, progress=True, cache=True)
    CFD = dict(zip(names, res))
else:
    CFD = {n: None for n in CFD_CASES}
CFDT = ncfd.table(CFD, P)
STATUS["CFD"] = "run" if RUN_CFD and (CFDT["status"] == "solved").any() else "NOT RUN"
display(save_table(CFDT, "cfd_results"))

In [ ]:
def cfd(name, key):
    r = CFDT.loc[name] if name in CFDT.index else None
    return float(r[key]) if r is not None and r["status"] == "solved" else np.nan

alphas = [a for a in ncfd.ALPHAS]
cl_cfd = [cfd(f"α {a:+.0f}°, 16 m/s", "Cl") for a in alphas]
cd_cfd = [cfd(f"α {a:+.0f}°, 16 m/s", "Cd") for a in alphas]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(POL.index, POL["CL"], "k-", label="lattice"); ax[0].plot(alphas, cl_cfd, "rs", label="CFD screening")
ax[1].plot(POL["CD"], POL["CL"], "k-", label="lattice + build-up Cd0"); ax[1].plot(cd_cfd, cl_cfd, "rs", label="CFD")
ax[2].plot(POL.index, POL["Cm (quarter MAC)"], "k-", label="lattice (quarter MAC)")
ax[2].plot(alphas, [cfd(f"α {a:+.0f}°, 16 m/s", "Cm (CG, MAC)") for a in alphas], "rs", label="CFD (CG)")
for a_, (xl, yl) in zip(ax, (("α [deg]", "CL"), ("CD", "CL"), ("α [deg]", "Cm"))):
    a_.set_xlabel(xl); a_.set_ylabel(yl); a_.grid(alpha=0.3); a_.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plots" / "polar_lattice_vs_cfd.png", dpi=120); plt.show()
cd_air, cd_wing = cfd("α +2°, 16 m/s", "Cd"), cfd("wing alone, α +2°, 16 m/s", "Cd")
cd_fine, cd_prop = cfd("α +2°, 16 m/s, fine mesh", "Cd"), cfd("α +2°, 16 m/s, propeller running", "Cd")
bu = nisus.drag_buildup(P, 16.0)
rest = sum(v for k, v in bu["parts_m2"].items() if k not in ("wing",)) / bu["planform"]
INTERF = pd.Series({"CFD: aircraft Cd at α 2°": cd_air, "CFD: wing alone Cd": cd_wing, "CFD: pod + booms + tail + interference (difference)": cd_air - cd_wing,
                    "build-up: the same share (incl. 10 % interference and extras)": rest,
                    "CFD: screening − fine (the screening's mesh error)": cd_air - cd_fine,
                    "CFD: airframe Cd with the propeller running − without (installation drag)": cd_prop - cd_air,
                    "build-up Cd0 (all)": bu["cd0"]}, name="Cd")
display(INTERF.to_frame())
CD0_CFD_CORRECTION = (cd_fine if np.isfinite(cd_fine) else cd_air) - (bu["cd0"] + nf.coefficients(DER["Zero"])["k"] * cfd("α +2°, 16 m/s", "Cl") ** 2) if np.isfinite(cd_air) else 0.0
print("Cd0 correction from the CFD (applied to later iterations, not to this one's simulation):", CD0_CFD_CORRECTION if np.isfinite(CD0_CFD_CORRECTION) else "NOT RUN")

# Part 8 — structure

**Load cases** from the flight conditions with the design mass (Zero + 10 %): the manoeuvre limit n = 4, Pratt's gusts
at cruise (7.5 m/s) and at V_NE = 28 m/s (3.75 m/s), the tail and fins at their section's maximum lift at V_NE, the hand
launch (12 m/s in 0.3 s, taken as 5 g), a belly landing (2.5 m/s sink over a 15 mm stroke of skid and grass), a
tail-first touchdown on the bumpers, the static thrust and torque. Ultimate = 1.5 × limit. Allowables carry
manufacturing knock-downs (printed parts 0.7, carbon 0.8, adhesives 0.5, foam 0.7).

In [ ]:
LC = nst.load_cases(P, pm=PM)
display(save_table(LC, "load_cases"))
display(save_table(nst.wing_hand(P, LC), "wing_by_hand"))
display(save_table(nst.joints_hand(P, LC), "joints_by_hand"))

**FEA** (Talos: Gmsh + CalculiX, C3D10, linear static): the main spar in the ultimate pull-up (right half, clamped in
the saddle), the rear carry-through tube and a boom in the tail case, the printed boom root fitting (the boom's bearing
reactions in its socket, held by its rings on the two tubes), the motor mount in flight and in a landing, the battery
tray. Materials are isotropic: the tubes with their axial modulus; printed parts with their in-plane values, then
checked **across the layers** (the build axis) against the layer-adhesion strength (60 % of in-plane, assumed). Peak
stresses at a fixed support's edge are singular: tube cases also report the nominal stress one diameter away.

In [ ]:
FEA_CASES = nst.models(ROOT / "fea", P, LC)
FEA = {}
for c in FEA_CASES:
    wd = ROOT / "fea" / c.name
    FEA[c.name] = c.model.ensure(wd, run=RUN_FEA, threads=4)
FEAT = pd.DataFrame([nst.summary_row(c, FEA[c.name]) for c in FEA_CASES]).set_index("case")
STATUS["FEA"] = "run" if (FEAT["status"] == "solved").any() else "NOT RUN"
display(save_table(FEAT.T, "fea_results"))

In [ ]:
r = FEA.get("boom_fitting")
if r is not None and r.ok:
    try:
        tviz.show(tviz.plot_results(r, field="von_mises"))
    except Exception as e:
        print("3D result view needs pyvista:", e)
else:
    print("boom_fitting: NOT RUN")

**Mesh convergence** — the boom root fitting (the printed part with the smallest margin) at four element sizes. A peak
that keeps growing with refinement is a singularity (a sharp re-entrant corner or the support edge), not a stress.

In [ ]:
if RUN_FEA:
    STEP_FIT = nst.export_step("boom_fitting_fea", ROOT / "fea" / "cad", P)
    CONV = nst.mesh_convergence(STEP_FIT, sizes=(3.0, 2.0, 1.4) if QUICK else (3.0, 2.0, 1.4, 1.0), workdir=ROOT / "fea" / "convergence", p=P, cases=LC)
    display(save_table(CONV, "mesh_convergence_boom_fitting"))
else:
    print("mesh convergence: NOT RUN")

In [ ]:
display(pd.DataFrame(nst.NOT_ESTABLISHED, columns=["not established by linear-static FEA", "what is missing"]).set_index("not established by linear-static FEA"))

# Part 9 — flight

Trim, CG range and static margin, control authority, stall margins, climb, coordinated turns and the bank-dependent
stall speed, the gust response, a bounded pull-up and the approach — from the linear model of Part 7 and the
propulsion map. These are first estimates; the CFD's polar and a flight test replace them. Neither the lattice nor
the few attached-flow CFD cases validate a stall or a recovery.

In [ ]:
AF = {v: nf.airframe(v) for v in VARIANTS}
UNIT = {v: nf.MapUnit(PM, fixed_w=float(ns.phase_power(v).loc["survey"].iloc[0])) for v in VARIANTS}
rows = []
for v in VARIANTS:
    for V in (10.0, 12.0, 14.0, 16.0, 18.0, 22.0, 28.0):
        t = nf.trim(DER[v], CI[v]["mass_kg"], V); t["variant"] = v; rows.append(t)
TRIM = pd.DataFrame(rows).set_index(["variant", "V"])
display(save_table(TRIM.round(3), "trim"))
CGR = {v: nf.cg_range(P, CI[v]["mass_kg"]) for v in VARIANTS}
display(pd.DataFrame(CGR).T.round(3))

In [ ]:
ctrl = {}
for v in VARIANTS:
    c = nf.coefficients(DER[v])
    Vs = AF[v].stall_speed()
    t_s = nf.trim(DER[v], CI[v]["mass_kg"], 1.05 * Vs)
    pb2v = c["Clda"] * math.radians(20) / -c["Clp"]
    beta_x = -c["Cndr"] * math.radians(25) / c["Cnb"]
    ctrl[v] = {"stall speed [m/s]": Vs, "elevator to trim at 1.05 Vs [deg]": t_s["elevator_deg"], "elevator travel [deg]": 25.0,
               "roll p b/2V at 20° aileron": pb2v, "roll rate at 16 m/s [deg/s]": math.degrees(pb2v * 2 * 16 / (P["span"] / 1000)),
               "steady sideslip at full rudder [deg]": math.degrees(beta_x), "max crosswind component at 1.3 Vs (that sideslip) [m/s]": 1.3 * Vs * math.sin(beta_x),
               "Cnβ (weathercock) [1/rad]": c["Cnb"], "Clβ (dihedral effect) [1/rad]": c["Clb"],
               "spiral stability Clβ Cnr − Cnβ Clr (> 0 stable)": c["Clb"] * c["Cnr"] - c["Cnb"] * c["Clr"]}
CTRL = pd.DataFrame(ctrl)
display(save_table(CTRL.round(3), "control_authority"))

In [ ]:
ENV = {v: nf.envelope(AF[v], UNIT[v], battery_wh=ns.battery("gens-ace-3s-2200").energy_wh) for v in VARIANTS}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for v in VARIANTS:
    e = ENV[v]
    ax[0].plot(e["V"], e["P_level"], color=COLOR[v], label=f"Nisus-{v} level flight"); ax[0].plot(e["V"], e["P_full"], color=COLOR[v], ls=":", lw=0.8)
    ax[1].plot(e["V"], e["roc"], color=COLOR[v], label=v); ax[2].plot(e["V"], e["radius_inst"], color=COLOR[v], label=v)
ax[0].set(xlabel="airspeed [m/s]", ylabel="electrical power [W]", ylim=(0, 300)); ax[1].set(xlabel="airspeed [m/s]", ylabel="climb rate, full throttle [m/s]")
ax[2].set(xlabel="airspeed [m/s]", ylabel="minimum turn radius [m]", ylim=(0, 60))
for a_ in ax: a_.grid(alpha=0.3); a_.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plots" / "envelope.png", dpi=120); plt.show()
perf = pd.DataFrame({v: {k: ENV[v][k] for k in ("v_stall", "v_endurance", "P_min", "endurance_min", "v_range", "range_km", "v_climb", "roc_max", "v_top",
                                              "L/D_max", "v_L/D_max", "sink_min_m_s", "r_min_inst", "rate_max_deg_s")} for v in VARIANTS})
display(save_table(perf.round(2), "performance"))

In [ ]:
for v in VARIANTS:
    print(f"Nisus-{v}: bank-dependent stall and turns"); display(nf.stall_table(AF[v]).round(2))
CHK = {}
for v in VARIANTS:
    g1, g2 = nf.gust_response(AF[v], DER[v], 16.0, 5.0), nf.gust_response(AF[v], DER[v], 16.0, 7.5)
    pu, ap = nf.pull_up(AF[v], 22.0), nf.approach(AF[v], UNIT[v])
    CHK[v] = {"gust 5 m/s at 16 m/s: n": g1["n_gust"], "gust 7.5 m/s at 16 m/s: n": g2["n_gust"], "stalls in the 7.5 m/s gust": g2["stalls_in_gust"],
              "pull-up at 22 m/s: n": pu["n"], "pull-up limited by": pu["limited_by"], "pull-up radius [m]": pu["radius_m"],
              "height lost recovering a 30° dive [m]": pu["height_lost_30deg_dive_m"], "corner speed V_A [m/s]": pu["V_A_corner_m_s"],
              "approach speed [m/s]": ap["V_approach"], "idle glide angle [deg]": ap["glide_angle_idle_deg"], "idle sink [m/s]": ap["sink_idle_m_s"],
              "6° approach: thrust [N]": ap["approach_thrust_N"], "touchdown speed [m/s]": ap["touchdown_speed_m_s"]}
display(save_table(pd.DataFrame(CHK), "flight_checks"))
LIM = {v: nf.operating_limits(v, AF[v], UNIT[v], DER[v], CGR[v]) for v in VARIANTS}
display(save_table(pd.concat({v: LIM[v]["value"] for v in VARIANTS}, axis=1).join(LIM["Zero"]["basis"]), "operating_limits"))

# Part 10 — mission energy and the return reserve

The sizing mission: 3 min of ground operation (electronics on, cameras and computer running, GNSS fix), a hand launch,
a 3 m/s climb to 80 m, the outbound leg to a field 600 m away, the survey (15 m/s, +10 % for the pattern's turns),
the return at 16 m/s, a 60 s approach and the landing — ten minutes airborne. The pack's available energy is
**derated** (90 %: temperature, age and the ESC's cut-off voltage — an assumption); 20 % of the available energy must
remain after landing. The return energy is **calculated from the route** (distance, wind along the way, cruise power,
electronics, the approach) plus **one go-around circuit** (900 m at cruise: the gusty simulation of Part 11 needed one, and
without it landed 0.5 Wh under the reserve), with a 15 % uncertainty allowance; the brief's 30 % allocation is shown for
comparison only.

Reference numbers of the brief: 3S 2200 mAh × 11.1 V = 24.42 Wh = 0.02442 kWh; 18 W for ten minutes = 3 Wh = 0.003
kWh = 12.3 % of it; 20 % of it = 4.884 Wh; 30 % = 7.326 Wh.

In [ ]:
b0 = ns.battery("gens-ace-3s-2200")
assert abs(b0.energy_wh - 24.42) < 1e-9 and abs(ns.JETSON_INSTALLATION_W * 600 / 3600 - 3.0) < 1e-9
print(f"{b0.energy_wh:.2f} Wh = {b0.energy_wh / 1000:.5f} kWh; 18 W x 10 min = {18 * 600 / 3600:.1f} Wh = {100 * 3.0 / b0.energy_wh:.1f} % of it; "
      f"20 % = {0.2 * b0.energy_wh:.3f} Wh; 30 % = {0.3 * b0.energy_wh:.3f} Wh")
def airframe_dict(v, bkey):
    t = ns.mass_table(v, bkey); return {"mass_kg": ns.cg_inertia(t)["mass_kg"], "cd0": BU["cd0"], "AR": L["AR"], "oswald": OSW, "S": L["S_ref"]}
PLAN = ns.MissionPlan()
ME = {v: ns.mission_energy(v, b0, PM, airframe_dict(v, b0.key), PLAN) for v in VARIANTS}
for v in VARIANTS:
    r = ME[v]
    print(f"\nNisus-{v} on the {b0.name}: used {r['E_used_wh']:.2f} Wh ({r['E_used_kwh']:.4f} kWh, {100 * r['E_used_wh'] / r['E_nominal_wh']:.1f} % of nominal), "
          f"left {r['E_left_pct_available']:.1f} % of the available {r['E_available_wh']:.2f} Wh -> reserve {'KEPT' if r['reserve_kept'] else 'NOT KEPT'}; "
          f"electronics {100 * r['electronics_share']:.0f} % of the energy")
    print(f"  return trigger: {r['trigger_wh']:.2f} Wh ({r['trigger_pct_nominal']:.1f} % of nominal) = return {r['return']['E_return_wh']:.2f} "
          f"(of which a go-around {r['return']['E_go_around_wh']:.2f}) + uncertainty "
          f"{r['return']['E_uncertainty_wh']:.2f} + reserve {r['E_reserve_wh']:.2f}; the brief's 30 % allocation would be {r['comparison_30pct_wh']:.2f} Wh")
    print(f"  survey in the ten minutes {r['t_survey_s']:.0f} s; feasible survey at this radius {r['t_survey_feasible_s'] / 60:.1f} min")
    display(save_table(r["phases"].round(4), f"mission_energy_{v}"))

In [ ]:
MT = {v: ns.mission_table(v, PM, airframe_dict, PLAN) for v in VARIANTS}
for v in VARIANTS:
    print(f"Nisus-{v}: the battery iteration (mass → power → endurance)"); display(save_table(MT[v].round(3), f"battery_iteration_{v}"))
sens = {}
for w in (15.0, 18.0, 20.0):
    r = ns.mission_energy("Zero", b0, PM, airframe_dict("Zero", b0.key), PLAN, jetson_w=w)
    sens[f"{w:g} W"] = {"used [Wh]": r["E_used_wh"], "left [% available]": r["E_left_pct_available"], "electronics share": r["electronics_share"],
                        "feasible survey [min]": r["t_survey_feasible_s"] / 60,
                        "energy-limited radius [km]": ns.mission_radius("Zero", b0, PM, airframe_dict("Zero", b0.key), PLAN, jetson_w=w, limit="energy") / 1000}
orin = ns.JETSON_OPTIONS["orin_nano_super"]
t_or = ns.mass_table("Zero", b0.key, jetson="orin_nano_super")
af_or = {"mass_kg": ns.cg_inertia(t_or)["mass_kg"], "cd0": BU["cd0"], "AR": L["AR"], "oswald": OSW, "S": L["S_ref"]}
r = ns.mission_energy("Zero", b0, PM, af_or, PLAN, jetson_w=orin["battery_w"])
sens["Orin Nano Super (substitution)"] = {"used [Wh]": r["E_used_wh"], "left [% available]": r["E_left_pct_available"], "electronics share": r["electronics_share"],
                                          "feasible survey [min]": r["t_survey_feasible_s"] / 60,
                                          "energy-limited radius [km]": ns.mission_radius("Zero", b0, PM, af_or, PLAN, jetson_w=orin["battery_w"], limit="energy") / 1000}
display(save_table(pd.DataFrame(sens).T.round(3), "jetson_sensitivity"))

# Part 11 — the missions in MuJoCo (Chiron)

A six-degree-of-freedom simulation: the aircraft is one rigid Chiron body carrying every mass-table row (MuJoCo's mass,
CG and inertia are the table's — checked below); the forces and moments come from the coefficient table of Part 7
through the `Aero` scene hook (the control surfaces with their servos' rate limit and lag, the propulsion map at the
lagged throttle along the tilted thrust line, wind and turbulence), gravity is MuJoCo's, its built-in fluid model is
off (asserted) so nothing is counted twice. The hand launch is an explicit impulse (m × 12 m/s over 0.3 s, 10° up)
from a held start; the ground contact is MuJoCo's soft contact on the keel skid and the tail bumpers (a simplified
model: no grass, no skid deformation). The motion — launch, climb, survey, the energy-triggered return, the approach
and the landing — emerges from the dynamics and the controller: Zero flies the computer's waypoints, OBS a scripted
pilot through the flight controller's stabilised modes, and the flight controller's energy manager calls the return.
The approach adds twice the measured airspeed fluctuation to the approach speed (the gust additive), trades excess
height for speed, goes around when it is still high over the aim point, and flares by holding a sink rate with at most
5° of nose-up pitch (the first gusty runs stalled in a harder flare and landed at 3.6 m/s: that is why).

In [ ]:
lab = nsc.make_lab(nsc.Scenario("Zero"))
IC = nr.inertia_check(lab, lab.robot.mass_table)
display(pd.Series(IC, name="MuJoCo vs the mass table").to_frame())
# force signs and the trim: the hook's coefficients at the linear trim of 16 m/s give lift = weight and no pitching moment
c = lab.coeff; t16 = nf.trim(lab.deriv, CI["Zero"]["mass_kg"], 16.0)
CL, CD, CY, Cl, Cm, Cn, st = lab.aero.coefficients(math.radians(t16["alpha_deg"]), 0.0, 0.0, 0.0, 0.0, np.array([0.0, math.radians(t16["elevator_deg"]), 0.0]), 16.0)
qS = 0.5 * RHO * 16 ** 2 * c["S"]
signs = {"lift / weight at the trim": qS * CL / (lab.ci["mass_kg"] * G), "Cm at the trim (≈ 0)": Cm, "CD > 0": CD > 0,
         "Cmα < 0 (stable)": c["Cma"] < 0, "Cnβ > 0 (weathercock)": c["Cnb"] > 0, "Clβ < 0 (dihedral)": c["Clb"] < 0, "Clp < 0 (roll damping)": c["Clp"] < 0,
         "Cmq < 0, Cnr < 0 (damping)": c["Cmq"] < 0 and c["Cnr"] < 0, "Clδa > 0: right-wing-down for a positive aileron": c["Clda"] > 0}
display(pd.Series(signs, name="force and moment signs").to_frame())

In [ ]:
# timestep sensitivity and energy accounting: the first 120 s of the Zero mission at two physics steps; the hook's
# energy (the ground operation charged at the start + its own sum of P dt) against an independent trapezoid of the
# logged power
STEP = {}
for ts in (0.002, 0.001):
    scn = nsc.Scenario("Zero", duration=120.0)
    lab_ = nsc.make_lab(scn, timestep=ts); ep_ = nsc.run(lab_, scn, duration=120.0)
    d = nsc.timeseries(ep_)
    ground = float(d["E_used_Wh"].iloc[0])
    P_int = float(np.trapezoid(d["P_el_W"], d["t"]) / 3600)
    STEP[f"{ts * 1000:g} ms"] = {"x at 120 s [m]": d["x"].iloc[-1], "y at 120 s [m]": d["y"].iloc[-1], "height at 120 s [m]": d["h"].iloc[-1],
                                 "airspeed at 120 s [m/s]": d["V"].iloc[-1], "energy used (hook) [Wh]": ep_.log["energy"]["E_used_wh"],
                                 "ground charge + trapezoid of logged P [Wh]": ground + P_int}
display(save_table(pd.DataFrame(STEP).round(4), "timestep_sensitivity"))

In [ ]:
SCN = [nsc.Scenario("OBS", "calm"), nsc.Scenario("Zero", "calm")] if QUICK else nsc.standard_scenarios()
EPS = []
for scn in SCN:
    t0 = time.time()
    lab_ = nsc.make_lab(scn)
    ep = nsc.run(lab_, scn)
    EPS.append((scn, ep))
    d = nsc.timeseries(ep); d.to_csv(OUT / "telemetry" / f"{scn.slug}.csv", index=False)
    print(f"{scn.label:40s} {time.time() - t0:5.0f} s  {ep.outcome['reason']:22s} {ep.outcome['detail']}")
    for e in ep.log["events"]:
        if e[1] in ("energy", "jetson", "failsafe", "landing") and "charged" not in e[2]:
            print(f"      t = {e[0]:7.1f} s  {e[1]}: {e[2]}")
COMP = nsc.compare([ep for _, ep in EPS])
display(save_table(COMP.T, "mission_comparison"))

In [ ]:
for scn, ep in EPS:
    print(scn.label); display(save_table(nsc.energy_table(ep).round(4), f"mission_energy_flown_{scn.slug}"))
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for scn, ep in EPS:
    d = nsc.timeseries(ep)
    ls = "-" if scn.variant == "Zero" else "--"
    ax[0].plot(d["t"], d["h"], ls, label=scn.label); ax[1].plot(d["t"], d["V"], ls, label=scn.label)
    ax[2].plot(d["t"], ep.log["energy"]["E_available_wh"] - d["E_used_Wh"], ls, label=scn.label)
ax[2].axhline(PLAN.reserve_frac * b0.energy_wh * PLAN.derating, color="k", lw=0.8, ls=":", label="reserve")
for a_, yl in zip(ax, ("height [m]", "airspeed [m/s]", "remaining energy [Wh]")):
    a_.set_ylabel(yl); a_.grid(alpha=0.3)
ax[0].legend(fontsize=7, ncol=2); ax[2].set_xlabel("t [s]")
fig.tight_layout(); fig.savefig(OUT / "plots" / "missions.png", dpi=120); plt.show()
fig, ax = plt.subplots(figsize=(9, 8))
for scn, ep in EPS:
    d = nsc.timeseries(ep); ax.plot(d["x"], d["y"], lw=0.8, label=scn.label)
ax.plot(0, 0, "k*", ms=12, label="home"); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(fontsize=7); ax.set(xlabel="x east [m]", ylabel="y north [m]")
fig.savefig(OUT / "plots" / "tracks.png", dpi=120); plt.show()

In [ ]:
# the movies: chase view (a ground camera near home), a synthetic onboard view, the map, the overlays
MOVIES = []
for scn, ep in EPS:
    path = OUT / f"{scn.slug}.mp4"
    try:
        nsc.render_movie(ep, scn, path, speed=25.0, fps=20, size=(640, 360) if QUICK else (960, 540))
        MOVIES.append(path); print("written", path)
    except Exception as e:
        print(f"{scn.label}: the movie needs pyvista with an off-screen renderer (xvfb-run on a headless machine): {type(e).__name__}: {e}")
if MOVIES:
    display(Video(str(MOVIES[0]), embed=False, width=800))

# Part 12 — the first recommended build

The tables below are generated from the numbers above (so they follow the next iteration's changes).

In [ ]:
def first_build(v):
    t, c_, me = TABLES[v], CI[v], ME[v]
    lim = LIM[v]["value"]
    flown = COMP.xs(v, level="variant") if v in COMP.index.get_level_values("variant") else None
    calm = flown.loc["calm"] if flown is not None and "calm" in flown.index else None
    return {
        "airframe": f"{P['span']:.0f} mm NACA 3412 wing, XPS core, 1.5 mm balsa D-box, film; 10/8 carbon spar + 10/8 rear tube; LW-PLA pod {P['pod_length']:.0f}×{P['pod_width']:.0f}×{P['pod_height']:.0f}; 10/8 carbon booms {P['boom_length']:.0f} mm; PETG fittings; TPU keel",
        "propulsion": "SunnySky X2216 V3 KV1250, APC 9×6E (pusher rotation), AM32 40 A ESC, 6° down-thrust",
        "battery": f"{ns.battery('gens-ace-3s-2200').name} ({b0.energy_wh:.2f} Wh, 190 g)",
        "flight control": "Holybro Kakute F405-Wing Mini, ArduPilot Plane (KakuteF405-Wing), Matek M10Q-5883, RadioMaster RP3 ELRS, SiK 433 MHz telemetry, 4 × Emax ES08MA II",
        "payload": "RunCam Phoenix 2 + RushFPV Tank Solo 5.8 GHz" + (" + Jetson Nano 4 GB B01 (JetPack 4.6.6), IMX219, 5 V/5 A buck, data link" if v == "Zero" else ""),
        "mass [g]": round(c_["mass_kg"] * 1000), "CG [% MAC]": round(100 * c_["x_cg_frac_mac"], 1),
        "static margin [% MAC]": round(100 * DER[v].loc["static_margin", "value"], 1), "wing loading [g/dm²]": round(c_["wing_loading_g_dm2"], 1),
        "stall / approach / cruise [m/s]": f"{lim['stall speed, level [m/s]']:.1f} / {lim['approach speed [m/s]']:.1f} / 16",
        "cruise power (propulsion + electronics) [W]": round(me["P_cruise_w"] + me["electronics_w"]["outbound"], 1),
        "ten-minute mission energy [Wh]": round(me["E_used_wh"], 2), "left after landing [% available]": round(me["E_left_pct_available"], 1),
        "electronics share of the energy": f"{100 * me['electronics_share']:.0f} %",
        "feasible survey at 600 m [min]": round(me["t_survey_feasible_s"] / 60, 1),
        "radius, ten-minute mission with a 2 min survey [km] (time-limited)": round(ns.mission_radius(v, b0, PM, airframe_dict(v, b0.key), PLAN) / 1000, 2),
        "radius, 2 min survey, energy-limited [km]": round(ns.mission_radius(v, b0, PM, airframe_dict(v, b0.key), PLAN, limit="energy") / 1000, 2),
        "simulated calm mission (airborne, left)": (f"{calm['airborne [s]'] / 60:.1f} min, {calm['remaining [% available]']:.0f} % left, {calm['reason']}" if calm is not None else "NOT RUN"),
        "limits for first flights": f"V_NE {lim['never-exceed speed [m/s]']:.0f} m/s, n ≤ {lim['limit load factor']:.0f}, bank ≤ 45°, wind ≤ 8 m/s, gusts ≤ 5 m/s, CG {lim['CG forward limit [% MAC]']:.0f}–{lim['CG aft limit [% MAC]']:.0f} % MAC",
    }
BUILD = pd.DataFrame({f"Nisus-{v}": first_build(v) for v in VARIANTS})
display(save_table(BUILD, "first_build"))

**Assumptions to refine next** (in order of what they move):

1. **Drag**: the flat-plate build-up's Cd0 against the CFD (fine mesh) and a glide test; the pod's after-body in the
   propeller's inflow (the rotor-disk case); laminar separation at Re ≈ 2·10⁵ (a transition model or a tunnel).
2. **Propulsion**: the X2216 + 9×6 on a thrust stand (static and in a car or a tunnel at 10–20 m/s) to replace the
   fitted motor model and the generic blade; the installed pusher's efficiency.
3. **Electronics power**: measure every branch with the flight controller's current sensor on the bench, the Jetson
   with the real vision pipeline (the 18 W installation figure is an estimate), the video transmitter at its set power.
4. **Battery**: the pack's usable energy at the ESC's cut-off and at 5 °C (the 90 % derating), its mass on a scale.
5. **Structure**: coupons of the printed PETG across the layers, the foam–balsa and the socket bonds; the carbon tube's
   modulus (catalogues give 28–150 GPa); a sandbag test of the wing to the limit load; the boom's first mode against
   the propeller's 1P/2P.
6. **Stability and control**: the lateral derivatives (assumed pod terms, Clp) and the stall from flight data; the CG
   range from the first flights (start at 25 % MAC).
7. **Component availability**: every component line is a search snippet; confirm stock and price before ordering —
   the Jetson Nano B01 developer kit is end-of-life: buy a module on a carrier or move to the Orin Nano (a substitution
   that changes the mass, the power and the software, see Part 10).

In [ ]:
STATUS.update({"CAD": "run", "drawings": "run", "mass budgets": "run", "electronics": "run", "propulsion (Boreas)": "run",
               "vortex lattice": "run", "mission energy": "run", "MuJoCo missions": f"run ({len(EPS)} scenarios)", "movies": f"{len(MOVIES)} written"})
if DEBUG:
    STATUS["FEA"] = STATUS["CFD"] = "MOCKED (VEGETA_DEBUG): the numbers mean nothing"
display(pd.Series(STATUS, name="status").to_frame())

In [ ]:
# the hand-off: the design's numbers for the next iteration and the scenario script (not written in DEBUG mode: mocked
# FEA and CFD numbers must never reach the design file)
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x
DESIGN = {"source": "31_nisus", "params": {k: plain(v) for k, v in P.items()}, "layout": {k: plain(v) for k, v in L.items()},
          "mass": {v: {k: plain(x) for k, x in CI[v].items() if k != "inertia"} for v in VARIANTS},
          "cd0_buildup": plain(BU["cd0"]), "cd0_cfd_correction": plain(CD0_CFD_CORRECTION) if np.isfinite(CD0_CFD_CORRECTION) else None,
          "derivatives": {v: {k: plain(x) for k, x in DER[v]["value"].items()} for v in VARIANTS},
          "propulsion_fit": {k: plain(x) for k, x in FIT.items() if not isinstance(x, str)},
          "mission_energy": {v: {k: plain(ME[v][k]) for k in ("E_used_wh", "E_left_pct_available", "reserve_kept", "electronics_share", "t_survey_feasible_s", "trigger_wh")} for v in VARIANTS},
          "fea": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in FEAT.T.to_dict().items()},
          "cfd": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in CFDT.T.to_dict().items()}, "status": STATUS}
target = OUT / "nisus_design.json" if DEBUG else Path("designs/data/nisus_design.json")
target.write_text(json.dumps(DESIGN, indent=1, default=str))
print("written", target)